# Modul 05 · nb08 — Deploy: RAG sebagai Layanan `/ask`

Notebook terakhir: kita **bungkus pipeline Ask-My-Document (nb07) menjadi sebuah layanan web**. Klien mana pun (web, mobile, skrip) bisa mengirim pertanyaan via HTTP dan menerima **jawaban + sumber** dalam JSON.

```
klien --HTTP POST /ask {"question": "..."}--> FastAPI
        -> RagEngine (retrieve + rerank + generate + verifikasi sitasi)
        <-- JSON {"answer": "... [S1]", "sources": [...], "cited": [1,3], "cited_invalid": []}
```

| Bagian | Isi |
|--------|-----|
| `RagEngine` | pipeline RAG dimuat **sekali** saat start, lalu melayani banyak pertanyaan |
| FastAPI `/ask` | endpoint POST dengan skema **Pydantic** (request & response tervalidasi) |
| Smoke test | `TestClient` **in-process** — uji endpoint tanpa menjalankan server |
| Runbook | cara menjalankan sungguhan (`uvicorn` + `curl`), env, cache, edge, checklist |

> **Kenapa tidak menjalankan server sungguhan di sel Colab?** Colab tidak dirancang untuk server yang hidup lama (sel akan menggantung). Jadi di sini kita pakai **`TestClient`** (memanggil app langsung di dalam proses, tanpa port) sebagai smoke test, dan **runbook** sebagai panduan deploy nyata.
>
> **Generator:** default **NVIDIA NIM** (cloud, tanpa GPU) — set `GENERATOR="qwen"` untuk Qwen2.5-3B lokal (T4). Cocok karena server biasanya tak punya GPU.

In [1]:
# Install dependencies
!pip install -q fastapi uvicorn httpx "pydantic>=2" "transformers<5" "sentence-transformers>=3.0" faiss-cpu accelerate bitsandbytes openai

import os, sys, json

REPO = "/content/navasena-gen-ml-course"
if not os.path.exists(REPO):
    !git clone https://github.com/chmdznr/navasena-gen-ml-course.git {REPO}
sys.path.append(os.path.abspath(f"{REPO}/05_rag"))

from tools.rag_utils import cited_labels
print("Import cited_labels OK ->", cited_labels("Contoh [S1] dan [S3].", 3))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.5 MB/s eta 0:00:00
Cloning into '/content/navasena-gen-ml-course'...
remote: Enumerating objects: 1384, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 1384 (delta 84), reused 92 (delta 58), pack-reused 1255 (from 1)
Receiving objects: 100% (1384/1384), 49.68 MiB | 28.86 MiB/s, done.
Resolving deltas: 100% (825/825), done.
Import cited_labels OK -> ([1, 3], [])


## 1. Mesin RAG — dimuat sekali

`RagEngine` menampung korpus + index + reranker + generator. Dalam layanan nyata, model dimuat **sekali saat start**, lalu setiap permintaan `/ask` hanya memanggil `engine.ask(...)` — tidak memuat ulang model.

Untuk demo, korpus = 12 pasal handbook karyawan (lihat nb06); tiap pasal adalah satu "sumber" yang bisa disitasi. Di produksi, kamu mengisi `passages`/`locators` dari hasil **ingest nb07** (chunk PDF + halaman/heading).

In [2]:
# ── Korpus + embedder + FAISS + reranker (komponen retrieval) ──
import faiss, numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

PASSAGES = [
    "Setiap pegawai tetap memperoleh jatah dua belas hari libur berbayar dalam satu tahun, terpisah dari hari libur nasional.",
    "Pengajuan cuti tahunan dilakukan melalui portal karyawan dan wajib mendapat persetujuan atasan langsung paling lambat tiga hari sebelumnya.",
    "Cuti sakit dapat diambil hingga sepuluh hari setiap tahun dan harus disertai surat keterangan dari dokter bila lebih dari dua hari berturut-turut.",
    "Karyawan baru menjalani masa orientasi selama tiga hari kerja pertama, mencakup pengenalan budaya perusahaan dan pelatihan keselamatan.",
    "Cuti melahirkan diberikan selama tiga bulan penuh sesuai ketentuan ketenagakerjaan yang berlaku di Indonesia.",
    "Jam kerja standar adalah pukul 09.00 hingga 17.00 dari Senin sampai Jumat, dengan satu jam istirahat untuk makan siang.",
    "Karyawan dapat bekerja dari rumah maksimal dua hari dalam seminggu setelah mendapat persetujuan dari manajer tim.",
    "Gaji dibayarkan pada tanggal 25 setiap bulan melalui transfer bank ke rekening masing-masing pegawai.",
    "Perusahaan menyediakan asuransi kesehatan yang mencakup rawat inap dan rawat jalan bagi pegawai beserta satu anggota keluarga.",
    "Lembur pada hari libur dihitung dua kali lipat dari tarif normal dan harus dicatat dalam sistem absensi sebelum dikerjakan.",
    "Pelanggaran kode etik dapat berujung pada surat peringatan hingga pemutusan hubungan kerja sesuai tingkat keparahannya.",
    "Penggantian biaya perjalanan dinas diajukan paling lambat tujuh hari setelah perjalanan dengan melampirkan bukti pembayaran yang sah.",
]
LOCATORS = [f"Pasal {i + 1}" for i in range(len(PASSAGES))]   # di produksi: ganti dgn "hlm N — heading" dari nb07

EMB_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMB_NAME)
embedder.max_seq_length = 512
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)
print("Embedder + reranker siap.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Embedder + reranker siap.


In [3]:
# ── Generator: "nim" (cloud, tanpa GPU — cocok untuk server) atau "qwen" (lokal, butuh GPU) ──
GENERATOR = "nim"

if GENERATOR == "nim":
    import getpass
    from openai import OpenAI                       # NVIDIA NIM kompatibel dengan API OpenAI
    try:
        from google.colab import userdata
        os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY") or ""
    except Exception:
        if not os.environ.get("NVIDIA_API_KEY"):
            os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA_API_KEY: ")
    assert os.environ.get("NVIDIA_API_KEY"), "GENERATOR='nim' butuh NVIDIA_API_KEY (Colab Secrets / env)."
    _client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["NVIDIA_API_KEY"])
    NIM_MODEL = "meta/llama-3.3-70b-instruct"
    def generate(prompt, max_new_tokens=256):
        r = _client.chat.completions.create(
            model=NIM_MODEL, temperature=0, max_tokens=max_new_tokens,
            messages=[{"role": "system", "content": "Jawab ringkas dalam Bahasa Indonesia."},
                      {"role": "user", "content": prompt}])
        return r.choices[0].message.content.strip()
    print("Generator: NVIDIA NIM ->", NIM_MODEL)
else:
    import warnings, torch
    warnings.filterwarnings("ignore", message=".*_check_is_size.*")
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    _name = "Qwen/Qwen2.5-3B-Instruct"
    tok = AutoTokenizer.from_pretrained(_name)
    _model = AutoModelForCausalLM.from_pretrained(_name, quantization_config=bnb, device_map="auto")
    for _k in ("temperature", "top_p", "top_k"):
        setattr(_model.generation_config, _k, None)
    def generate(prompt, max_new_tokens=256):
        msgs = [{"role": "system", "content": "Jawab ringkas dalam Bahasa Indonesia."},
                {"role": "user", "content": prompt}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors="pt").to(_model.device)
        out = _model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
        return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print("Generator: Qwen2.5-3B lokal ->", _model.device)

Generator: NVIDIA NIM -> meta/llama-3.3-70b-instruct


In [4]:
# ── RagEngine: retrieve + rerank + generate + verifikasi sitasi, dibungkus rapi ──
class RagEngine:
    def __init__(self, passages, locators, embedder, reranker, generate):
        self.passages, self.locators = passages, locators
        self.embedder, self.reranker, self.generate = embedder, reranker, generate
        cvecs = embedder.encode(passages, convert_to_numpy=True).astype("float32")
        self.index = faiss.IndexFlatL2(cvecs.shape[1]); self.index.add(cvecs)

    def retrieve(self, query, k_over=12, k_top=3):
        qv = self.embedder.encode([query], convert_to_numpy=True).astype("float32")
        _, idx = self.index.search(qv, min(k_over, len(self.passages))); cand = idx[0].tolist()
        scores = self.reranker.predict([(query, self.passages[c]) for c in cand]).tolist()
        order = sorted(range(len(cand)), key=lambda i: scores[i], reverse=True)[:k_top]
        return [cand[i] for i in order]

    def ask(self, question):
        ids = self.retrieve(question)
        blocks = [f"[S{n}] ({self.locators[c]}) {self.passages[c]}" for n, c in enumerate(ids, 1)]
        answer = self.generate(f"Konteks (tiap sumber berlabel [S#]):\n" + "\n\n".join(blocks) +
                               f"\n\nPertanyaan: {question}\n\nJawab HANYA berdasarkan konteks dan bubuhkan "
                               f"label [S#] pada klaim yang relevan. Bila konteks tidak memuat jawabannya, "
                               f"katakan informasinya tidak tersedia.")
        valid, invalid = cited_labels(answer, len(ids))
        return {"answer": answer,
                "sources": [{"label": f"S{n}", "sumber": self.locators[c]} for n, c in enumerate(ids, 1)],
                "cited": valid, "cited_invalid": invalid}

engine = RagEngine(PASSAGES, LOCATORS, embedder, reranker, generate)
print("RagEngine siap. Uji langsung:")
print(json.dumps(engine.ask("Berapa hari cuti sakit?"), indent=2, ensure_ascii=False))

RagEngine siap. Uji langsung:
{
  "answer": "Cuti sakit dapat diambil hingga sepuluh hari setiap tahun [S1].",
  "sources": [
    {
      "label": "S1",
      "sumber": "Pasal 3"
    },
    {
      "label": "S2",
      "sumber": "Pasal 5"
    },
    {
      "label": "S3",
      "sumber": "Pasal 1"
    }
  ],
  "cited": [
    1
  ],
  "cited_invalid": []
}


## 2. Bungkus jadi API FastAPI `/ask`

**Pydantic** mendefinisikan bentuk request & response — FastAPI otomatis memvalidasi input dan menjamin output sesuai skema (klien tahu persis apa yang akan diterima). Dua endpoint:
- `GET /health` — cek layanan hidup + jumlah sumber.
- `POST /ask` — kirim `{"question": "..."}`, terima jawaban + sumber + label sitasi yang terverifikasi.

In [5]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

class AskRequest(BaseModel):
    question: str

class SourceItem(BaseModel):
    label: str
    sumber: str

class AskResponse(BaseModel):
    answer: str
    sources: List[SourceItem]
    cited: List[int]            # label [S#] yang dipakai & valid
    cited_invalid: List[int]    # label di luar daftar (sitasi mengada-ada) — idealnya kosong

app = FastAPI(title="Ask-My-Document", version="1.0")

@app.get("/health")
def health():
    return {"status": "ok", "sources": len(engine.passages), "generator": GENERATOR}

@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    return engine.ask(req.question)

print("FastAPI app + endpoint /health dan /ask siap.")

FastAPI app + endpoint /health dan /ask siap.


## 3. Smoke test in-process (`TestClient`)

`TestClient` memanggil aplikasi **langsung di dalam proses** — tanpa membuka port, tanpa server yang menggantung. Inilah cara aman menguji endpoint **di dalam Colab**. (Untuk server sungguhan, lihat runbook di bawah.)

In [6]:
from fastapi.testclient import TestClient
client = TestClient(app)

print("GET /health ->", client.get("/health").json(), "\n")

for q in ["Berapa hari cuti sakit?", "Kapan gaji dibayarkan?", "Berapa lama cuti melahirkan?"]:
    resp = client.post("/ask", json={"question": q})
    print(f"POST /ask  {{'question': '{q}'}}  ->  HTTP {resp.status_code}")
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
    print("-" * 70)

# Validasi kontrak API: input cacat ditolak Pydantic (HTTP 422)
bad = client.post("/ask", json={"salah": "field"})       # field 'question' hilang
print("Request tak valid -> HTTP", bad.status_code, "(422 = Pydantic menolak input cacat)")

GET /health -> {'status': 'ok', 'sources': 12, 'generator': 'nim'} 

POST /ask  {'question': 'Berapa hari cuti sakit?'}  ->  HTTP 200
{
  "answer": "Cuti sakit dapat diambil hingga sepuluh hari setiap tahun [S1].",
  "sources": [
    {
      "label": "S1",
      "sumber": "Pasal 3"
    },
    {
      "label": "S2",
      "sumber": "Pasal 5"
    },
    {
      "label": "S3",
      "sumber": "Pasal 1"
    }
  ],
  "cited": [
    1
  ],
  "cited_invalid": []
}
----------------------------------------------------------------------
POST /ask  {'question': 'Kapan gaji dibayarkan?'}  ->  HTTP 200
{
  "answer": "Gaji dibayarkan pada tanggal 25 setiap bulan [S1].",
  "sources": [
    {
      "label": "S1",
      "sumber": "Pasal 8"
    },
    {
      "label": "S2",
      "sumber": "Pasal 1"
    },
    {
      "label": "S3",
      "sumber": "Pasal 6"
    }
  ],
  "cited": [
    1
  ],
  "cited_invalid": []
}
----------------------------------------------------------------------
POST /ask  {'ques

## 4. Runbook — menjalankan layanan sungguhan

Di Colab kita berhenti di smoke test (server hidup akan menggantung sel). Di mesin sendiri / server / edge, jalankan sebagai server HTTP nyata.

Daripada menyalin sel satu per satu (rawan lupa import), **sel berikut menulis satu berkas `ask_service.py` yang sudah lengkap & mandiri** — semua import ada, `cited_labels` di-inline, kunci dibaca dari environment, tanpa kode khusus Colab.

In [7]:
%%writefile ask_service.py
# ask_service.py — layanan Ask-My-Document yang berdiri sendiri (generator: NVIDIA NIM, tanpa GPU).
# Jalankan:  export NVIDIA_API_KEY=...   &&   uvicorn ask_service:app --host 0.0.0.0 --port 8000
import os, re
from typing import List
import numpy as np, faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from fastapi import FastAPI
from pydantic import BaseModel

# Korpus demo (12 pasal handbook). Di produksi: isi dari hasil ingest nb07 (chunk PDF + halaman/heading).
PASSAGES = [
    "Setiap pegawai tetap memperoleh jatah dua belas hari libur berbayar dalam satu tahun, terpisah dari hari libur nasional.",
    "Pengajuan cuti tahunan dilakukan melalui portal karyawan dan wajib mendapat persetujuan atasan langsung paling lambat tiga hari sebelumnya.",
    "Cuti sakit dapat diambil hingga sepuluh hari setiap tahun dan harus disertai surat keterangan dari dokter bila lebih dari dua hari berturut-turut.",
    "Karyawan baru menjalani masa orientasi selama tiga hari kerja pertama, mencakup pengenalan budaya perusahaan dan pelatihan keselamatan.",
    "Cuti melahirkan diberikan selama tiga bulan penuh sesuai ketentuan ketenagakerjaan yang berlaku di Indonesia.",
    "Jam kerja standar adalah pukul 09.00 hingga 17.00 dari Senin sampai Jumat, dengan satu jam istirahat untuk makan siang.",
    "Karyawan dapat bekerja dari rumah maksimal dua hari dalam seminggu setelah mendapat persetujuan dari manajer tim.",
    "Gaji dibayarkan pada tanggal 25 setiap bulan melalui transfer bank ke rekening masing-masing pegawai.",
    "Perusahaan menyediakan asuransi kesehatan yang mencakup rawat inap dan rawat jalan bagi pegawai beserta satu anggota keluarga.",
    "Lembur pada hari libur dihitung dua kali lipat dari tarif normal dan harus dicatat dalam sistem absensi sebelum dikerjakan.",
    "Pelanggaran kode etik dapat berujung pada surat peringatan hingga pemutusan hubungan kerja sesuai tingkat keparahannya.",
    "Penggantian biaya perjalanan dinas diajukan paling lambat tujuh hari setelah perjalanan dengan melampirkan bukti pembayaran yang sah.",
]
LOCATORS = [f"Pasal {i + 1}" for i in range(len(PASSAGES))]


def cited_labels(answer, n):
    # Verifikasi label [S#]: kembalikan (valid 1..n, invalid di luar rentang).
    nums = sorted({int(m) for m in re.findall(r"\[S(\d+)\]", answer)})
    return [x for x in nums if 1 <= x <= n], [x for x in nums if not 1 <= x <= n]


_client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["NVIDIA_API_KEY"])

def generate(prompt, max_new_tokens=256):
    r = _client.chat.completions.create(
        model="meta/llama-3.3-70b-instruct", temperature=0, max_tokens=max_new_tokens,
        messages=[{"role": "system", "content": "Jawab ringkas dalam Bahasa Indonesia."},
                  {"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip()


class RagEngine:
    def __init__(self, passages, locators):
        self.passages, self.locators = passages, locators
        self.embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        self.embedder.max_seq_length = 512
        self.reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)
        cv = self.embedder.encode(passages, convert_to_numpy=True).astype("float32")
        self.index = faiss.IndexFlatL2(cv.shape[1]); self.index.add(cv)

    def retrieve(self, q, k_over=12, k_top=3):
        qv = self.embedder.encode([q], convert_to_numpy=True).astype("float32")
        _, idx = self.index.search(qv, min(k_over, len(self.passages))); cand = idx[0].tolist()
        sc = self.reranker.predict([(q, self.passages[c]) for c in cand]).tolist()
        return [cand[i] for i in sorted(range(len(cand)), key=lambda i: sc[i], reverse=True)[:k_top]]

    def ask(self, q):
        ids = self.retrieve(q)
        blocks = [f"[S{n}] ({self.locators[c]}) {self.passages[c]}" for n, c in enumerate(ids, 1)]
        prompt = ("Konteks (tiap sumber berlabel [S#]):\n" + "\n\n".join(blocks) +
                  f"\n\nPertanyaan: {q}\n\nJawab HANYA berdasarkan konteks dan bubuhkan label [S#] pada klaim "
                  "yang relevan. Bila konteks tidak memuat jawabannya, katakan informasinya tidak tersedia.")
        answer = generate(prompt)
        valid, invalid = cited_labels(answer, len(ids))
        return {"answer": answer,
                "sources": [{"label": f"S{n}", "sumber": self.locators[c]} for n, c in enumerate(ids, 1)],
                "cited": valid, "cited_invalid": invalid}


engine = RagEngine(PASSAGES, LOCATORS)


class AskRequest(BaseModel):
    question: str

class SourceItem(BaseModel):
    label: str
    sumber: str

class AskResponse(BaseModel):
    answer: str
    sources: List[SourceItem]
    cited: List[int]
    cited_invalid: List[int]


app = FastAPI(title="Ask-My-Document", version="1.0")

@app.get("/health")
def health():
    return {"status": "ok", "sources": len(engine.passages)}

@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    return engine.ask(req.question)


Writing ask_service.py


### Jalankan & panggil

```bash
# 1. (sekali) pasang dependensi server
pip install fastapi uvicorn "sentence-transformers>=3.0" faiss-cpu openai

# 2. set kunci NIM lalu jalankan server
export NVIDIA_API_KEY=nvapi-xxxxx
uvicorn ask_service:app --host 0.0.0.0 --port 8000

# 3. panggil dari klien mana pun
curl -s -X POST http://localhost:8000/ask \
  -H "Content-Type: application/json" \
  -d '{"question": "Berapa hari cuti sakit?"}' | python3 -m json.tool
```

**Catatan deploy:**
- **Cache model:** set `HF_HOME=/path/cache` agar bobot model tidak diunduh ulang tiap start. Siapkan **~3 GB** disk (reranker `bge-reranker-v2-m3` ~2.2 GB + embedder ~0.5 GB). Model dimuat **sekali** di `RagEngine.__init__`, bukan per-request.
- **Tanpa GPU:** dengan NIM, hanya embedder + reranker yang lokal. Embedder ringan; **reranker (568 jt parameter) bisa jalan di CPU tapi menambah latensi per-request** — untuk edge yang ketat, pakai reranker lebih kecil atau lewati rerank (lihat nb03).
- **Edge / on-prem GPU:** untuk generator lokal, ganti blok generator dengan path `qwen` (butuh tambahan `pip install torch transformers bitsandbytes accelerate` + GPU CUDA — 4-bit tidak jalan di CPU), atau arahkan `base_url` NIM ke microservice NIM yang kamu host sendiri.
- **⚠️ Keamanan:** demo ini **tanpa auth & rate limit** — jangan diekspos ke publik apa adanya. Untuk produksi tambahkan API key/JWT di header, **rate limit** (mis. `slowapi`), dan batasi CORS. Jangan pernah hardcode `NVIDIA_API_KEY` — selalu dari environment/secrets.
- **Skalabilitas:** untuk korpus besar, ganti `IndexFlatL2` dengan index FAISS persisten (nb05: `IndexHNSWFlat` / IVF + `write_index`); ganti pembangunan index di `__init__` dengan `faiss.read_index(path)` saat start.

## 5. Checklist deploy & penutup Modul 05

### ✅ Checklist deploy
- [ ] Model dimuat **sekali** saat start (bukan per-request)
- [ ] `NVIDIA_API_KEY` dari **environment/secrets**, bukan hardcode
- [ ] `HF_HOME` di-set agar cache model awet
- [ ] Endpoint `/health` untuk health check (load balancer)
- [ ] Validasi input via **Pydantic** (request cacat → HTTP 422)
- [ ] **Verifikasi sitasi** (`cited_invalid` kosong) sebelum jawaban dipercaya
- [ ] Auth + rate limit + CORS untuk produksi
- [ ] Index FAISS persisten untuk korpus besar (nb05)

### Perjalanan Modul 05 — RAG
| nb | Topik |
|----|-------|
| 01 | Fundamentals RAG (embed → retrieve → generate) |
| 02 | Ingest & chunk (Docling, strategi chunking) |
| 03 | Retrieve lebih baik (bi-encoder → cross-encoder rerank) |
| 04 | Mengukur kualitas (RAGAS, judge NIM) |
| 05 | Menskalakan index (FAISS IVF/HNSW, ChromaDB) |
| 06 | Conversational RAG (memori + history-aware rewriting) |
| 07 | Capstone: Ask-My-Document (jawaban bersitasi terverifikasi) |
| **08** | **Deploy: RAG sebagai layanan `/ask`** |

Dari satu pertanyaan-jawaban sederhana sampai layanan RAG yang bisa di-deploy dengan jawaban yang **bisa ditelusuri sumbernya**. 🎓